# Microsoft OmniParser v2.0 - Universal UI Icon Detector

> **No training needed!** OmniParser is completely pre-trained by Microsoft.
>
> **Important:** Ensure your Colab runtime is set to **T4 GPU**:
> Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` -> `Save`.

### Step 1: Clone OmniParser Repository

In [ ]:
!git clone https://github.com/microsoft/OmniParser.git
%cd OmniParser

### Step 2: Install Dependencies

In [ ]:
# Install dependencies specified in the repository
!pip install -r requirements.txt
!pip install huggingface_hub

### Step 3: Download Pre-Trained Weights from Hugging Face
OmniParser v2 uses a fine-tuned YOLO model for icon detection and a fine-tuned Florence-2 model for semantic icon captioning.

In [ ]:
import os
from huggingface_hub import snapshot_download

# Create local weights folder and download pre-trained weights
os.makedirs("weights", exist_ok=True)
snapshot_download(
    repo_id="microsoft/OmniParser-v2.0",
    local_dir="weights"
)

print("Weights downloaded successfully:", os.listdir("weights"))

### Step 4: Load OmniParser Models into GPU

In [ ]:
import torch
from PIL import Image
from utils import check_ocr_box, get_caption_model_processor, get_som_labeled_img, get_yolo_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 1. Load YOLO Icon Detection Model
som_model = get_yolo_model(model_path="weights/icon_detect/best.pt")
som_model.to(device)

# 2. Load Florence-2 Icon Captioner
caption_model_processor = get_caption_model_processor(
    model_name="florence2", 
    model_name_or_path="weights/icon_caption"
)
print("OmniParser models loaded successfully!")

### Step 5: Define Universal Icon Detection Function

In [ ]:
def parse_screen_and_find_icons(image_path: str, search_keywords: list = None):
    """
    Parses a screenshot with OmniParser and optionally filters for target icons.
    """
    # 1. OCR text detection
    ocr_text, ocr_bbox = check_ocr_box(
        image_path,
        display_img=False,
        output_bb_format="xyxy",
        goal_filtering=None,
        use_paddleocr=True,
    )

    # 2. Parse all UI icons and labels
    labeled_img, label_coords, parsed_content_list = get_som_labeled_img(
        image_path,
        som_model,
        BOX_TRESHOLD=0.05,
        output_coord_in_ratio=True,
        ocr_bbox=ocr_bbox,
        caption_model_processor=caption_model_processor,
        ocr_text=ocr_text,
        use_local_semantics=True,
        iou_threshold=0.7,
        scale_img=False,
        batch_size=128,
    )

    # Display labeled image with bounding boxes & ID numbers
    display(Image.fromarray(labeled_img))

    print(f"Total UI elements detected: {len(parsed_content_list)}\n")

    # 3. Filter for specific icons (e.g. play, like, share, comment)
    matches = []
    for idx, element in enumerate(parsed_content_list):
        # element typically has format: 'icon: <description>' or 'text: <words>'
        element_text = str(element).lower()
        
        if search_keywords:
            if any(keyword.lower() in element_text for keyword in search_keywords):
                coord = label_coords.get(idx, "N/A")
                matches.append((idx, element, coord))
                print(f"MATCH FOUND [Box #{idx}]: {element} -> Normalized Coordinates: {coord}")
        else:
            print(f"[Box #{idx}]: {element}")

    return matches, parsed_content_list

### Step 6: Test on an Image
You can test using sample images from the repo (`imgs/google_page.png`), or upload your own screenshots of YouTube, TikTok, or Instagram!

In [ ]:
# Example 1: Test with included sample screenshot
test_image = "imgs/google_page.png"

# Search for specific icons
keywords = ["play", "search", "share", "heart", "like", "menu"]
matches, all_elements = parse_screen_and_find_icons(test_image, search_keywords=keywords)

### Step 7: Upload Your Own App Screenshot (YouTube, TikTok, Instagram)
Run this cell to upload any screenshot from your computer and detect all buttons!

In [ ]:
from google.colab import files

uploaded = files.upload()
for filename in uploaded.keys():
    print(f"Analyzing uploaded screenshot: {filename}")
    parse_screen_and_find_icons(filename, search_keywords=["play", "like", "heart", "share", "comment", "bookmark"])